# BirdCLEF+ 2025 - Final ensemble: submission notebook (CNN noisy student + Perch A2d)

**CPU only, internet off.** `VARIANT` in the first code cell selects the pre-registered variant:

| Variant | Models (weights) | Temporal smoothing |
|---|---|---|
| E1 | MAIN noisy student 0.5, Perch A2 + A1-distilled head 0.5 | no |
| E2 | as E1 | yes |
| E3 | MAIN 1/3, A2d 1/3, ABLATION noisy student 1/3 | yes |
| E4 | MAIN 0.4, A2d 0.6 | yes |

Each soundscape is decoded once per file and the audio is shared by all models. Four spawned worker processes handle whole files with one thread each. The CNN code (A1 V2-S, librosa mel, `zscore_minmax`) is the code of `ns-student-main-infer`, and the Perch code (resample to 32 kHz, peak 0.25, ONNX, head) is the code of `a2-a1distill-infer`.

**Smoothing** (per model and file, on probabilities, before ranking): `p'_t = 0.25 p_{t-1} + 0.5 p_t + 0.25 p_{t+1}`, and for the first/last window `(0.5 p_t + 0.25 p_nb) / 0.75`.

**Combination:** each model's class column is ranked over all rows (average ties) and scaled to [0, 1] by `(r - 1) / (N - 1)`, and then the ranks are averaged with the variant weights.

When `test_soundscapes/` is empty, the notebook does a dry run on 700 training soundscapes and projects 700 files against the 75 minute budget. Sanity checks (columns, rows, NaN, range, correlation with each single model) are written to `ens_checks.json`, and the notebook stops if one fails.

## Configuration and inputs

In [ ]:
VARIANT = "E2"          # "E1", "E2", "E3" or "E4": the only line that differs between the four kernels

import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"                # set before numpy loads; each worker uses one core
import time
T_NOTEBOOK = time.time()

import torch
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = "cpu"
print("torch.cuda.is_available():", CUDA_AVAILABLE)
print("DEVICE:", DEVICE)
if CUDA_AVAILABLE:
    raise RuntimeError("A GPU is visible: this submission must run on CPU. Run this notebook without an accelerator.")

import glob, hashlib, importlib, importlib.metadata, json, multiprocessing as mp, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

VARIANT = os.environ.get("M3_VARIANT", VARIANT)     # local tests only
VARIANTS = {   # pre-registered in STATE.md before any ensemble result existed
    "E1": dict(weights={"main": 0.5, "a2d": 0.5}, smooth=False),
    "E2": dict(weights={"main": 0.5, "a2d": 0.5}, smooth=True),
    "E3": dict(weights={"main": 1 / 3, "a2d": 1 / 3, "ablation": 1 / 3}, smooth=True),
    "E4": dict(weights={"main": 0.4, "a2d": 0.6}, smooth=True),
}
if VARIANT not in VARIANTS:
    raise ValueError(f"unknown VARIANT {VARIANT}")
WEIGHTS, SMOOTH = VARIANTS[VARIANT]["weights"], VARIANTS[VARIANT]["smooth"]
USE_ABLATION = "ablation" in WEIGHTS

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1" or os.environ.get("M3_SMOKE_TEST") == "1"   # local only
ORT_VERSION = "1.31.1"
NUM_WORKERS = int(os.environ.get("M3_INFER_WORKERS", "4"))      # 0 = no Pool
DRY_FILES = int(os.environ.get("M3_DRY_FILES", "700"))
BUDGET_MIN = 75
WINDOWS_PER_FILE = 12
MIN_CORR = 0.5
PERCH_SHA256 = "bf0c8467a924cb074663970ca4a0ab1e143602121930209657d0dff5d5cefa1f"
WORK = Path(os.environ.get("M3_WORK_DIR", "."))
WORK.mkdir(parents=True, exist_ok=True)

def find_parent(marker, env, fallback):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    return Path(hits[0]).parent if hits else Path(fallback)

def find_model_dir(kernel, env):
    """Both student kernels hold a model.pth, so match on the kernel name."""
    if os.environ.get(env):                 # local tests only
        return Path(os.environ[env])
    hits = [h for h in glob.glob("/kaggle/input/**/model.pth", recursive=True) if kernel in h]
    if len(hits) != 1:
        raise RuntimeError(f"expected one model.pth under kernel {kernel}, found {hits}")
    return Path(hits[0]).parent

def load_cnn_cfg(model_dir):
    cfg = json.load(open(model_dir / "cfg.json"))
    cfg.setdefault("norm", "v2"); cfg.setdefault("norm_top_db", 80.0)
    if not cfg["model"].startswith("tf_efficientnetv2_s"):
        raise RuntimeError(f"expected tf_efficientnetv2_s in {model_dir}, cfg.json has {cfg['model']}")
    return cfg

DATA = find_parent("birdclef-2025/sample_submission.csv", "M3_DATA_DIR", "../../data")
sample_sub = pd.read_csv(DATA / "sample_submission.csv")
SUB_COLUMNS = list(sample_sub.columns)
species = SUB_COLUMNS[1:]

# CNN students
CNN = {"main": find_model_dir("sc4000-birdclef-m3-ns-student-main", "M3_MAIN_DIR")}
if USE_ABLATION:
    CNN["ablation"] = find_model_dir("sc4000-birdclef-m3-ns-student-ablation", "M3_ABL_DIR")
CNN_CFG = {k: load_cnn_cfg(d) for k, d in CNN.items()}
for k, d in CNN.items():
    assert json.load(open(d / "species.json")) == species, f"{k}: species order differs from sample_submission"

# Perch A2 + A1-distilled head
HEAD_NAME = "perch_head_a1distill.npz"          # exact file from a2-a1distill, never the embed kernel's perch_head.npz
HEAD_DIR = find_parent(HEAD_NAME, "M3_HEAD_DIR", "../a2_a1distill")
if not (HEAD_DIR / HEAD_NAME).is_file():
    raise FileNotFoundError(f"{HEAD_NAME} not found in {HEAD_DIR}: attach strippedbanana/sc4000-birdclef-m3-a2-a1distill")
assert json.load(open(HEAD_DIR / "species.json")) == species, "a2d: species order differs from sample_submission"
ONNX_PATH = Path(os.environ["M3_PERCH_ONNX"]) if os.environ.get("M3_PERCH_ONNX") \
    else find_parent("perch_v2.onnx", "M3_ONNX_DIR", ".") / "perch_v2.onnx"
assert ONNX_PATH.exists(), f"perch_v2.onnx not found: attach strippedbanana/sc4000-birdclef-m3-perch-onnx ({ONNX_PATH})"

# Only install from the attached wheels if the image lacks this exact version. Nothing imports
# onnxruntime in this process, so the workers are the first to load it.
try:
    have = importlib.metadata.version("onnxruntime")
except importlib.metadata.PackageNotFoundError:
    have = None
wheels = find_parent("ort_cpu_wheels", "M3_WHEELS_DIR", "../a2_perch_embed") / "ort_cpu_wheels"   # the embed kernel
if have != ORT_VERSION and not SMOKE_TEST:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--force-reinstall",
                        "--find-links", str(wheels), f"onnxruntime=={ORT_VERSION}"], capture_output=True, text=True)
    if r.returncode != 0:
        if have is None:
            raise RuntimeError(f"onnxruntime is missing and no wheel fits this Python: {r.stderr[-400:]}")
        print(f"Keeping image onnxruntime {have}; wheel install failed: {r.stderr[-200:]}")
print("onnxruntime image version:", have, "| wanted:", ORT_VERSION)

print(f"VARIANT {VARIANT} | weights {WEIGHTS} | smoothing {SMOOTH} | workers {NUM_WORKERS} | CPUs {os.cpu_count()}")
print("Data:", DATA, "| CNN:", {k: str(d) for k, d in CNN.items()}, "| head:", HEAD_DIR, "| onnx:", ONNX_PATH)
print("CNN cfg:", {k: (c["model"], c["norm"]) for k, c in CNN_CFG.items()})
print("Head:", json.load(open(HEAD_DIR / "head_cfg.json"))["best_head"])

## ONNX checksum and file list

In [ ]:
t0 = time.time()
if not SMOKE_TEST:
    h = hashlib.sha256()
    with open(ONNX_PATH, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    if h.hexdigest() != PERCH_SHA256:
        raise RuntimeError(f"perch_v2.onnx checksum mismatch: {h.hexdigest()}")
SHA_S = time.time() - t0

test_files = sorted((DATA / "test_soundscapes").glob("*.ogg"))
DRY_RUN = len(test_files) == 0
if DRY_RUN:
    test_files = sorted((DATA / "train_soundscapes").glob("*.ogg"))[:DRY_FILES]
if not test_files:
    raise RuntimeError(f"no soundscapes found under {DATA}")
print(f"{'DRY RUN on' if DRY_RUN else 'Predicting'} {len(test_files)} soundscapes | sha check {SHA_S:.1f}s")
if DRY_RUN and len(test_files) < DRY_FILES:
    print(f"WARNING: only {len(test_files)} of {DRY_FILES} requested files exist")

## Worker code
We write this code to a module so that `spawn` workers can import it. One decode per file feeds every model.

In [ ]:
LIB_SRC = r'''
import os, time
import numpy as np
import soundfile as sf
import librosa
from scipy.signal import resample_poly

# ---------------- CNN students: same code as ns-student-main-infer (torch backend) ----------------
def to_input(db, cfg):
    # Same as to_input in the A1 notebooks. For norm "v2" this equals M1's (db + 80) / 80.
    top_db = float(cfg["norm_top_db"])
    db = np.maximum(db - db.max(), -top_db)
    if cfg["norm"] == "zscore_minmax":
        x = (db - db.mean()) / (db.std() + 1e-6)
        return ((x - x.min()) / (x.max() - x.min() + 1e-6)).astype(np.float32)
    return ((db + top_db) / top_db).astype(np.float32)

def cnn_windows(y, cfg):
    win = cfg["sr"] * cfg["duration"]
    n_win = max(1, int(np.ceil(len(y) / win)))
    y = np.pad(y, (0, n_win * win - len(y)))
    return y.reshape(n_win, win)

def windows_to_inputs(batch, cfg):
    # One melspectrogram call for the file. Each row gets its own STFT, as in the per-window loop.
    power = librosa.feature.melspectrogram(
        y=batch, sr=cfg["sr"], n_fft=cfg["n_fft"], hop_length=cfg["hop_length"],
        n_mels=cfg["n_mels"], fmin=cfg["fmin"], fmax=cfg["fmax"])
    # ref=np.max and the normalisation stay per window.
    top_db = float(cfg["norm_top_db"])
    return np.stack([to_input(librosa.power_to_db(m, ref=np.max, top_db=top_db), cfg) for m in power])

MEL_KEYS = ("sr", "duration", "n_fft", "hop_length", "n_mels", "fmin", "fmax", "norm", "norm_top_db")

def build_torch(cfg, n_classes, weights):
    import timm, torch
    model = timm.create_model(cfg["model"], pretrained=False, in_chans=1, num_classes=n_classes)
    model.load_state_dict(torch.load(weights, map_location="cpu"))
    return model.eval()

# ---------------- Perch A2d: same code as a2-a1distill-infer ----------------
SR, WIN_SEC = 32_000, 5
WIN = SR * WIN_SEC          # 160000 samples, the fixed Perch 2.0 input

def to_32k_mono(y, sr):
    """Shared with the other A2 notebook, keep both copies identical."""
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR:
        y = resample_poly(y, SR, sr)     # polyphase, as perch-hoplite does
    return y.astype(np.float32)

def preprocess_windows(x):
    """Perch preprocessing, per window: remove the mean, scale the peak to 0.25.
    Shared with the other A2 notebook, keep both copies identical."""
    x = x - x.mean(axis=1, keepdims=True)
    peak = np.abs(x).max(axis=1, keepdims=True)
    return (x * (0.25 / np.maximum(peak, 1e-8))).astype(np.float32)

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Shared with the other A2 notebook."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

# ---------------- worker ----------------
_cnn, _sess, _head, _init, _init_end = {}, None, None, {}, 0.0

def init_worker(cnn_specs, n_classes, onnx_path, head_path):
    """cnn_specs: list of (name, cfg, model.pth). Every model gets one thread."""
    global _sess, _head, _init_end
    import torch
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass        # already set in this process
    for name, cfg, weights in cnn_specs:
        t0 = time.perf_counter()
        _cnn[name] = (cfg, build_torch(cfg, n_classes, weights))
        _init[name] = time.perf_counter() - t0
    t0 = time.perf_counter()
    import onnxruntime as ort          # first load happens inside the worker
    so = ort.SessionOptions()
    so.intra_op_num_threads = 1
    so.inter_op_num_threads = 1
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    _sess = ort.InferenceSession(str(onnx_path), so, providers=["CPUExecutionProvider"])
    assert _sess.get_inputs()[0].name == "inputs"
    _head = dict(np.load(head_path))
    _init["a2d"] = time.perf_counter() - t0
    _init_end = time.time()

def predict_file(path):
    import torch
    st = {}
    t0 = time.perf_counter()
    y, sr = sf.read(str(path), dtype="float32")     # the one decode, shared by all models
    if y.ndim > 1:
        y = y.mean(axis=1)
    st["decode"] = time.perf_counter() - t0
    probs, row_ids = {}, None

    # CNN students. The mel is computed once per distinct mel config (once when both students share it).
    mels = {}
    st["cnn_mel"] = 0.0
    for name, (cfg, model) in _cnn.items():
        t0 = time.perf_counter()
        key = tuple(cfg[k] for k in MEL_KEYS)
        if key not in mels:
            yc = y if sr == cfg["sr"] else resample_poly(y, cfg["sr"], sr).astype(np.float32)
            mels[key] = windows_to_inputs(cnn_windows(yc, cfg), cfg)
        X = mels[key]
        t1 = time.perf_counter()
        with torch.inference_mode():
            logits = model(torch.from_numpy(X)[:, None]).numpy()
        probs[name] = torch.sigmoid(torch.from_numpy(logits)).numpy().astype(np.float32)
        st["cnn_mel"] += t1 - t0
        st[f"model_{name}"] = time.perf_counter() - t1
        ids = [f"{path.stem}_{(i + 1) * cfg['duration']}" for i in range(len(X))]
        if row_ids is not None and ids != row_ids:
            raise RuntimeError(f"{path.name}: window layout differs between models")
        row_ids = ids

    # Perch A2d
    t0 = time.perf_counter()
    y32 = to_32k_mono(y, sr)
    n_win = max(1, int(np.ceil(len(y32) / WIN)))
    y32 = np.pad(y32, (0, n_win * WIN - len(y32)))
    x = preprocess_windows(np.stack([y32[i * WIN:(i + 1) * WIN] for i in range(n_win)]))
    t1 = time.perf_counter()
    emb = _sess.run(["embedding"], {"inputs": x})[0]       # the 12 windows of a file are one batch
    t2 = time.perf_counter()
    probs["a2d"] = head_predict(_head, emb).astype(np.float32)
    st.update(perch_preprocess=t1 - t0, model_a2d=t2 - t1, a2d_head=time.perf_counter() - t2)
    ids = [f"{path.stem}_{(i + 1) * WIN_SEC}" for i in range(n_win)]
    if row_ids is not None and ids != row_ids:
        raise RuntimeError(f"{path.name}: window layout differs between CNN and Perch")
    st.update(init=dict(_init), init_end=_init_end, pid=os.getpid())
    return ids, probs, st
'''
(WORK / "ens_infer_lib.py").write_text(LIB_SRC)
sys.path.insert(0, str(WORK.resolve()))
importlib.invalidate_caches()
import ens_infer_lib as lib

## Predict

In [ ]:
cnn_specs = [(k, CNN_CFG[k], CNN[k] / "model.pth") for k in CNN]
initargs = (cnn_specs, len(species), ONNX_PATH, HEAD_DIR / HEAD_NAME)
PIPELINE_T0 = time.time()
SETUP_S = PIPELINE_T0 - T_NOTEBOOK
if NUM_WORKERS > 0:
    ctx = mp.get_context("spawn")
    with ctx.Pool(NUM_WORKERS, initializer=lib.init_worker, initargs=initargs) as pool:
        results = list(pool.imap(lib.predict_file, test_files, chunksize=1))
else:
    lib.init_worker(*initargs)
    results = [lib.predict_file(p) for p in test_files]
PIPELINE_WALL_S = time.time() - PIPELINE_T0
MODEL_NAMES = list(results[0][1])
print("Models run:", MODEL_NAMES, f"| {len(results)} files in {PIPELINE_WALL_S:.1f}s")

## Smoothing, ranking and checks (functions)

In [ ]:
# Ensemble functions. Pure functions of their arguments (unit-tested locally on synthetic arrays).
import numpy as np
from scipy.stats import rankdata

def smooth_file(p):
    """p: (windows, classes) of ONE file, windows in time order.
    Interior: 0.25 p[t-1] + 0.5 p[t] + 0.25 p[t+1]; first/last: (0.5 p[t] + 0.25 p[nb]) / 0.75."""
    p = np.asarray(p, dtype=np.float64)
    n = len(p)
    if n == 1:
        return p.copy()
    out = 0.5 * p
    out[1:] += 0.25 * p[:-1]
    out[:-1] += 0.25 * p[1:]
    out[0] /= 0.75
    out[-1] /= 0.75
    return out

def rank01(P):
    """Per-class rank over all rows (average ties), scaled to [0, 1] by (r - 1) / (N - 1); N == 1 -> 0.5."""
    P = np.asarray(P, dtype=np.float64)
    n = P.shape[0]
    if n == 1:
        return np.full(P.shape, 0.5)
    return (rankdata(P, axis=0, method="average") - 1.0) / (n - 1.0)

def rank_average(probs, weights):
    """probs: {model: (rows, classes)}; weights: {model: w}. Weighted mean of the models' rank01."""
    total = float(sum(weights.values()))
    return sum(w * rank01(probs[m]) for m, w in weights.items()) / total

def colwise_pearson(A, B):
    A = A - A.mean(axis=0); B = B - B.mean(axis=0)
    den = np.sqrt((A * A).sum(axis=0) * (B * B).sum(axis=0))
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, (A * B).sum(axis=0) / np.where(den > 0, den, 1.0), np.nan)

def correlations(ens, P):
    """Per-class Spearman averaged over classes (constant columns skipped), overall Pearson
    against the model's ranks (the quantity that is averaged) and, for information, against its probabilities."""
    sp = colwise_pearson(rankdata(ens, axis=0), rankdata(P, axis=0))
    return dict(spearman_per_class_mean=float(np.nanmean(sp)) if np.isfinite(sp).any() else float("nan"),
                n_classes_spearman=int(np.isfinite(sp).sum()),
                pearson_vs_ranks=float(np.corrcoef(ens.ravel(), rank01(P).ravel())[0, 1]),
                pearson_vs_probs=float(np.corrcoef(ens.ravel(), np.asarray(P, np.float64).ravel())[0, 1]))

def run_checks(sub, sub_columns, n_files, windows_per_file, probs, ens, min_corr):
    """Returns (report, failures). Checks columns, row count, unique row ids, NaN/inf, [0, 1], correlation."""
    fails = []
    vals = sub[sub_columns[1:]].to_numpy(dtype=np.float64) if list(sub.columns) == list(sub_columns) else None
    rep = dict(columns_match=list(sub.columns) == list(sub_columns),
               n_rows=int(len(sub)), expected_rows=int(windows_per_file * n_files),
               row_ids_unique=bool(sub["row_id"].is_unique),
               all_finite=bool(vals is not None and np.isfinite(vals).all()),
               in_0_1=bool(vals is not None and np.isfinite(vals).all() and vals.min() >= 0.0 and vals.max() <= 1.0),
               models={})
    if not rep["columns_match"]: fails.append("columns differ from sample_submission")
    if rep["n_rows"] != rep["expected_rows"]: fails.append(f"rows {rep['n_rows']} != {rep['expected_rows']}")
    if not rep["row_ids_unique"]: fails.append("duplicate row_id")
    if not rep["all_finite"]: fails.append("NaN or inf in submission")
    if not rep["in_0_1"]: fails.append("values outside [0, 1]")
    for m, P in probs.items():
        c = correlations(ens, P)
        c["probs_finite"] = bool(np.isfinite(P).all())
        c["probs_in_0_1"] = bool(np.isfinite(P).all() and P.min() >= 0 and P.max() <= 1)
        rep["models"][m] = c
        if not c["probs_finite"] or not c["probs_in_0_1"]: fails.append(f"{m}: probabilities not finite or outside [0, 1]")
        for k in ("spearman_per_class_mean", "pearson_vs_ranks"):
            if not (c[k] >= min_corr):      # also catches NaN
                fails.append(f"{m}: {k} {c[k]:.3f} < {min_corr}")
    rep["failures"] = fails
    rep["passed"] = not fails
    return rep, fails

## Ensemble

In [ ]:
t0 = time.time()
row_ids = [r for ids, _, _ in results for r in ids]
# Smoothing is per model and per file (each result is one file, windows in time order), before ranking.
# float32 so that the ranked arrays are exactly the saved .npy files.
probs = {m: np.concatenate([smooth_file(p[m]) if SMOOTH else p[m] for _, p, _ in results]).astype(np.float32)
         for m in MODEL_NAMES}
missing = [m for m in WEIGHTS if m not in probs]
if missing:
    raise RuntimeError(f"variant {VARIANT} needs {missing}, not run")
ens = rank_average(probs, WEIGHTS)
for m, P in probs.items():
    np.save(WORK / f"probs_{m}{'_smoothed' if SMOOTH else ''}.npy", P)
pd.Series(row_ids, name="row_id").to_csv(WORK / "probs_row_ids.csv", index=False)
POST_S = time.time() - t0
print(f"Ensemble {VARIANT}: {ens.shape} from {list(WEIGHTS)} | smoothing {SMOOTH} | post-processing {POST_S:.1f}s")

## Timing
Stage seconds are summed over workers; `timing_profile.json` has the details.

In [ ]:
n = len(test_files)
stat = [r[2] for r in results]
stage = lambda k: sum(s.get(k, 0.0) for s in stat)
workers = {s["pid"]: s for s in stat}
n_active = len(workers)
STAGE_KEYS = ["decode", "cnn_mel"] + [f"model_{m}" for m in MODEL_NAMES] + ["perch_preprocess", "a2d_head"]
STAGES = {k: stage(k) for k in STAGE_KEYS}
LOAD = {m: max(s["init"][m] for s in workers.values()) for m in MODEL_NAMES}
# Elapsed time until the slowest reporting worker was ready (spawn, imports, model load).
STARTUP_WALL_S = min(max(max(s["init_end"] for s in workers.values()) - PIPELINE_T0, 0.0), PIPELINE_WALL_S)
# Parallel work cannot beat the summed compute time divided by the workers, which bounds short runs.
per_file_wall = max((PIPELINE_WALL_S - STARTUP_WALL_S) / n, sum(STAGES.values()) / n_active / n)
fixed_s = SETUP_S + STARTUP_WALL_S + POST_S * 700 / n
projected_min = (fixed_s + per_file_wall * 700) / 60
gate = "OK" if projected_min <= BUDGET_MIN else "FAIL"

profile = {
    "variant": VARIANT, "weights": WEIGHTS, "smoothing": SMOOTH,
    "source": "train_soundscapes (dry run)" if DRY_RUN else "test_soundscapes",
    "n_files_used": n, "n_windows": len(row_ids), "num_workers": NUM_WORKERS, "active_workers": n_active,
    "threads_per_worker": 1, "cpu_count": os.cpu_count(), "device": DEVICE, "cuda_available": CUDA_AVAILABLE,
    "stages_s": {**{k: round(v, 4) for k, v in STAGES.items()},
                 "setup_incl_sha": round(SETUP_S, 4), "sha_check": round(SHA_S, 4),
                 "worker_startup_elapsed": round(STARTUP_WALL_S, 4), "post_processing": round(POST_S, 4),
                 "total_pipeline": round(PIPELINE_WALL_S, 4), "total_notebook": round(time.time() - T_NOTEBOOK, 4)},
    "model_load_s": {m: round(v, 4) for m, v in LOAD.items()},
    "per_file_mean_s": {k: round(v / n, 6) for k, v in STAGES.items()},
    "per_file_elapsed_s": round(per_file_wall, 6),
    "projected_700_files_min": round(projected_min, 2), "budget_min": BUDGET_MIN, "gate": gate,
    "timing_note": "stage seconds are summed over workers (compute seconds); per_file_elapsed_s is wall time per file",
}
with open(WORK / "timing_profile.json", "w") as f:
    json.dump(profile, f, indent=2)
print("TIMING | " + " | ".join(f"{k} {v:.1f}s" for k, v in STAGES.items()) + f" (summed over {n_active} workers)")
print(f"TIMING | setup {SETUP_S:.1f}s (sha {SHA_S:.1f}s) | worker startup {STARTUP_WALL_S:.1f}s "
      f"(load {', '.join(f'{m} {v:.1f}s' for m, v in LOAD.items())}) | post {POST_S:.1f}s | {per_file_wall:.2f}s per file elapsed")
print(f"TIMING | projected 700 files: {projected_min:.1f} min (budget {BUDGET_MIN}) -> {gate} | measured on {n} files")

## Sanity checks and `submission.csv`

In [ ]:
sub = pd.DataFrame(ens.astype(np.float32), columns=species)
sub.insert(0, "row_id", row_ids)
report, failures = run_checks(sub, SUB_COLUMNS, len(test_files), WINDOWS_PER_FILE, probs, ens, MIN_CORR)
report.update(variant=VARIANT, weights=WEIGHTS, smoothing=SMOOTH, dry_run=DRY_RUN, min_corr=MIN_CORR)
with open(WORK / "ens_checks.json", "w") as f:
    json.dump(report, f, indent=2)
print(f"SANITY | columns {report['columns_match']} | rows {report['n_rows']}/{report['expected_rows']} | "
      f"unique ids {report['row_ids_unique']} | finite {report['all_finite']} | in [0,1] {report['in_0_1']}")
for m, c in report["models"].items():
    print(f"SANITY | ensemble vs {m}: Spearman per class {c['spearman_per_class_mean']:.3f} ({c['n_classes_spearman']} classes) | "
          f"Pearson vs ranks {c['pearson_vs_ranks']:.3f} | Pearson vs probs {c['pearson_vs_probs']:.3f} (info)")
print(f"SANITY | {'PASS' if report['passed'] else 'FAIL: ' + '; '.join(failures)}")
if failures:
    raise RuntimeError("sanity checks failed: " + "; ".join(failures))

sub.to_csv(WORK / "submission.csv", index=False)
print("submission.csv", sub.shape, "| variant", VARIANT)
if DRY_RUN:
    print("NOTE: submission.csv covers training soundscapes; this was a dry run.")
print(f"TOTAL notebook time: {(time.time() - T_NOTEBOOK) / 60:.2f} min")
sub.iloc[:3, :4]